# Lunar Lander RL Lab — Reinforcement Learning & Reward Engineering

Welcome to the **Lunar Lander RL Lab** interactive laboratory. In this notebook, we explore:
1. **LunarLander-v3 Environment Mechanics**: Observation space (8D) and action space (4 discrete).
2. **Untrained Random Policy Baseline**: Performance metrics of random actions.
3. **Reward Engineering & Reward Hacking**: Comparing Baseline, Flawed State-Bonus, and Corrected Progress Shaping.
4. **Training with Stable-Baselines3 PPO**: Interactive training and real-time metric logging.
5. **Comparative Evaluation & Gap Analysis**: Distinguishing perceived training reward from true landing performance.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import gymnasium as gym

# Ensure project root is in sys.path
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.lunar_rl.environment.factory import make_lunar_environment
from src.lunar_rl.environment.observation import LunarObservation, OBSERVATION_DESCRIPTIONS, ACTION_DESCRIPTIONS
from src.lunar_rl.environment.random_agent import run_random_agent
from src.lunar_rl.rewards.reward_wrapper import wrap_environment
from src.lunar_rl.agents.ppo_agent import PPOAgent
from src.lunar_rl.evaluation.evaluator import Evaluator

print("Lunar Lander RL Lab modules loaded successfully!")


## 1. Inspecting the LunarLander-v3 Environment & 8D Observations
LunarLander-v3 provides an 8-dimensional continuous observation vector and 4 discrete engine actions.

In [ ]:
env = make_lunar_environment(seed=42)
obs, info = env.reset(seed=42)
parsed_obs = LunarObservation.from_array(obs)

print("=== LunarLander-v3 Initial State ===")
print(parsed_obs.explain())
print("
Observation Dimensions Breakdown:")
obs_df = pd.DataFrame(OBSERVATION_DESCRIPTIONS)
obs_df


## 2. Baseline Benchmark: Untrained Random Agent
Let us run the random agent to observe baseline score distribution.

In [ ]:
random_summary = run_random_agent(episodes=3, seed=42, log_observations=False)
pd.DataFrame([random_summary])


## 3. Reward Engineering: Baseline vs Flawed (Hacking) vs Corrected
We observe how custom reward wrappers transform native step rewards.

In [ ]:
# Create a reward hacking environment
hack_cfg = {"type": "reward_hacking", "upright_bonus": 0.5, "velocity_bonus": 0.5}
hacked_env = wrap_environment(make_lunar_environment(seed=42), hack_cfg)

obs, _ = hacked_env.reset(seed=42)
obs, r, term, trunc, info = hacked_env.step(0)

print("Reward Hacking Step Info:")
print(f"  Native Reward: {info['original_reward']:.4f}")
print(f"  Shaped Reward: {info['shaped_reward']:.4f}")
print(f"  Components:    {info['reward_components']}")
hacked_env.close()


## 4. Evaluation and Visualizations
We can use the  to assess any policy in pure native LunarLander-v3.

In [ ]:
evaluator = Evaluator()
print("Evaluator ready for full cross-regime benchmark.")
